# Building a synthetic NER dataset for mountain names

This notebook walks through the step-by-step process of creating a synthetic, tokenized Named Entity Recognition (NER) dataset in the IOB (Inside-Outside-Beginning) format.

### Dataset Structure

The dataset contains two main columns:

- tokens: List of tokenized words
- tags: Corresponding NER tags

NER tags:

- Tag B-MOUNTAIN: Beginning of a mountain entity
- Tag I-MOUNTAIN: Inside a multi-token mountain entity
- Tag O: Non-mountain tokens


## Step 1: Raw text generation (LLM generation)

In the first stage, a Gemini is used to generate natural English sentences containing names of mountains and mountain ranges.

Output file format (`unlabeled_data.json`):
```json
[
  {
    "text": "Mount Rainier dominates the skyline of western Washington state.",
    "mountains": ["Mount Rainier"]
  }
]
```


## Step 2: Automatic tokenization and tagging

The popular IOB (Inside, Outside, Beginning) scheme is used for NER:
* `B-MOUNTAIN`: The first token of the mountain name (Beginning).
* `I-MOUNTAIN`: Subsequent tokens of the mountain name, if it consists of multiple words (Inside).
* `O`: All other words and punctuation marks (Outside).



In [1]:
import json
import re
import os
from collections import Counter


def tokenize_and_tag(text, mountain_names=None):
    """
    Tokenizes raw text and tags mountain entities using the IOB scheme.
    """
    if mountain_names is None:
        mountain_names = []

    tokens = re.findall(r'\w+|[^\w\s]', text)
    tags = ['O'] * len(tokens)

    sorted_mountains = sorted(mountain_names, key=lambda x: len(re.findall(r'\w+|[^\w\s]', x)), reverse=True)

    for mnt in sorted_mountains:
        mnt_tokens = re.findall(r'\w+|[^\w\s]', mnt)
        mnt_len = len(mnt_tokens)

        for i in range(len(tokens) - mnt_len + 1):
            sub = tokens[i:i + mnt_len]
            if [t.lower() for t in sub] == [mt.lower() for mt in mnt_tokens]:
                if all(tags[j] == 'O' for j in range(i, i + mnt_len)):
                    tags[i] = 'B-MOUNTAIN'
                    for j in range(i + 1, i + mnt_len):
                        tags[j] = 'I-MOUNTAIN'

    return {"tokens": tokens, "tags": tags}

print("Environment setup complete!")

## Step 3: Running the tagging and deduplication pipeline

The script reads `unlabeled_data.json`, checks the existing `ner_mountain_dataset.json`, and appends only new instances by comparing token tuples.

In [3]:
unlabeled_file = "data/unlabeled_data.json"
dataset_file = "data/ner_mountain_dataset.json"

with open(unlabeled_file, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

existing_dataset = []
if os.path.exists(dataset_file):
    with open(dataset_file, "r", encoding="utf-8") as f:
        try:
            existing_dataset = json.load(f)
            print(f"Loaded existing dataset containing {len(existing_dataset)} records.")
        except json.JSONDecodeError:
            existing_dataset = []

existing_tokens = {tuple(entry["tokens"]) for entry in existing_dataset}
new_tagged_entries = []

for item in raw_data:
    text = item.get("text", "")
    mountains = item.get("mountains", [])
    tagged_sample = tokenize_and_tag(text, mountains)

    sample_tokens = tuple(tagged_sample["tokens"])
    if sample_tokens not in existing_tokens:
        new_tagged_entries.append(tagged_sample)
        existing_tokens.add(sample_tokens)

final_dataset = existing_dataset + new_tagged_entries

with open(dataset_file, "w", encoding="utf-8") as f:
    json.dump(final_dataset, f, indent=2, ensure_ascii=False)

print(f"Tagged and appended {len(new_tagged_entries)} new samples.")
print(f"Total entries in '{dataset_file}': {len(final_dataset)}")

Loaded existing dataset containing 700 records.
Tagged and appended 1 new samples.
Total entries in 'data/ner_mountain_dataset.json': 701


## Dataset Visualization and Analysis

Check the structure of the resulting dataset and print the tag balance

In [4]:
with open(dataset_file, "r", encoding="utf-8") as f:
    dataset = json.load(f)

for idx, entry in enumerate(dataset):
    assert len(entry["tokens"]) == len(entry["tags"]), f"Sequence length mismatch at index {idx}"

print("Validation passed: Token and tag lengths match across all samples!\n")

tag_counts = Counter()
for entry in dataset:
    tag_counts.update(entry["tags"])

print("--- Tag Distribution ---")
for tag, count in tag_counts.items():
    print(f"{tag:12s}: {count}")

print("\n--- Sample Output Preview ---")
print(json.dumps(dataset[0], indent=2, ensure_ascii=False))

Validation passed: Token and tag lengths match across all samples!

--- Tag Distribution ---
B-MOUNTAIN  : 518
I-MOUNTAIN  : 491
O           : 7418

--- Sample Output Preview ---
{
  "tokens": [
    "Mount",
    "Rainier",
    "dominates",
    "the",
    "skyline",
    "of",
    "western",
    "Washington",
    "state",
    "."
  ],
  "tags": [
    "B-MOUNTAIN",
    "I-MOUNTAIN",
    "O",
    "O",
    "O",
    "O",
    "O",
    "O",
    "O",
    "O"
  ]
}
